In [1]:
%%capture
# Updated: January 2026 - Latest package versions
# Note: If you've already run install.sh, these packages are already installed
# Fixed typo: removed extra dash in package name
!pip install llama-index==0.14.13 llama-index-embeddings-cohere==0.6.1

In [2]:
# Standard library imports
import os
from getpass import getpass
import nest_asyncio

# Third-party imports
from dotenv import load_dotenv

# Apply nest_asyncio to allow nested event loops (needed for Jupyter notebooks)
# This is required when using async operations in Jupyter
nest_asyncio.apply()

# Load environment variables from .env file
# This will read CO_API_KEY and other variables from the .env file in the project root
load_dotenv()

True

In [3]:
# Get Cohere API key from environment variable
# Falls back to prompting user if not found in .env file
# Note: Using os.getenv() is safer than os.environ[] as it returns None instead of raising KeyError
CO_API_KEY = os.getenv("CO_API_KEY") or getpass("Enter your Cohere API key: ")

# 🗂️ Indexing

An `Index` is a data structure that allows for the quick retrieval of relevant context for a user query. 

It is the core foundation for retrieval-augmented generation (RAG) use-cases. Indexes are built from `Documents` and are used to build Retrievers, Query Engines and Chat Engines. All of which enable question & answer and chat over your data.

- 📂 After loading your data, you're ready to construct an `Index`.

- 🌐 **Vector Store Index:** The most common Index type. It segments your `Documents` into `Nodes` and generates vector embeddings for each node's text, prepping them for LLM queries.

- 🔄 **Vector Store Index Process:** Parse raw texts into document objects, split document objects into chunks/nodes, then convert all your nodes into embeddings and store them in a vector database.

### ⚙️ Embedding Text

First, let's see what an embedding is.


In [4]:
# Import CohereEmbedding from LlamaIndex
# Embeddings convert text into numerical vectors that capture semantic meaning
# Similar texts have similar vectors, enabling semantic search
from llama_index.embeddings.cohere import CohereEmbedding

# Initialize different Cohere embedding models
# Each model has different characteristics:
# - embed-english-v3.0: Latest, highest quality, 1024 dimensions
# - embed-english-light-v3.0: Faster, lighter, 384 dimensions (good for speed/cost)
# - embed-english-v2.0: Older model, 4096 dimensions (deprecated but still works)
# Note: Always pass api_key explicitly or it will read from environment
embed_v3 = CohereEmbedding(
    api_key=CO_API_KEY,
    model_name="embed-english-v3.0"  # Latest model: 1024 dimensions, best quality
)

embed_v3_light = CohereEmbedding(
    api_key=CO_API_KEY,
    model_name="embed-english-light-v3.0"  # Lightweight: 384 dimensions, faster/cheaper
)

embed_v2 = CohereEmbedding(
    api_key=CO_API_KEY,
    model_name="embed-english-v2.0"  # Older model: 4096 dimensions (deprecated)
) 

#### You can also use local embedding models, by using an embedding model from Hugging Face. Check the [MTEB Leaderboard for what's hot](huggingface.co/spaces/mteb/leaderboard)

```python

pip install llama-index-embeddings-huggingface

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

hf_embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
```

#### If you're running locally and on a CPU, though, you may want to use `FastEmbed`. These models are lightweight, quantized, and optimized for CPU. Here are the [supported models](https://qdrant.github.io/fastembed/examples/Supported_Models/)

This is how you can instantiate a `FastEmbed` model:

```python
pip install llama-index-embeddings-fastembed

from llama_index.embeddings.fastembed import FastEmbedEmbedding

embed_model = FastEmbedEmbedding(model_name="BAAI/bge-large-en-v1.5-quantized")
```

In [6]:
string = "A"

string_2 = "This is a complete sentence."

string_3 = """In the pursuit of a life well-lived, one must recognize the transient nature of the 
material world and the enduring value of virtue. The Sikh Gurus taught us that the Divine Light 
resides within all, and thus, we are united in our essence beyond the superficial distinctions of 
caste, creed, or status. Similarly, the Stoics emphasized the cultivation of inner virtues such as courage, 
temperance, and wisdom, understanding that true freedom lies in mastery over one's own perceptions and actions. 
As we navigate the vicissitudes of life, let us remember that our choices are our own, and in choosing virtue, 
we align ourselves with the cosmic order and the teachings of the Gurus. It is through selfless service, 
compassion, and the relentless pursuit of truth that we may attain a state of inner peace and contribute 
to the harmony of the world, embodying the principles of both Sikhism and Stoicism in our daily lives
"""

In [7]:
example_embedding = embed_v3.get_text_embedding(string)

In [8]:
# Check the dimensionality of the embedding vector
# embed-english-v3.0 produces 1024-dimensional vectors
# Higher dimensions can capture more nuanced semantic information
len(example_embedding)

1024

In [ ]:
def get_embedding_dimensions(embed_model, list_of_strings):
    embeddings = embed_model.get_text_embedding_batch(list_of_strings)   
    # print(embeddings)
    embed_lens = []
    for embedding in embeddings:
        embed_lens.append(len(embedding))
    return embed_lens

In [12]:
get_embedding_dimensions(embed_v3, [string, string_2, string_3])

[[0.010307312, -0.010192871, -0.02746582, -0.06774902, 0.055480957, -0.009635925, -0.029724121, -0.009849548, 0.035064697, 0.009185791, -0.024505615, 0.04525757, -0.04269409, -0.010009766, 0.012550354, 0.009651184, -0.0041542053, 0.015563965, -0.019012451, -0.033081055, -0.0345459, -0.03945923, -0.0060653687, 0.016113281, 0.013145447, 0.0052452087, -0.03475952, 0.012268066, -0.049865723, -0.0039749146, -0.009529114, 0.05355835, 0.02519226, 0.011390686, 0.013664246, -0.015548706, -0.03479004, -0.003686905, -0.014968872, 0.017059326, 0.00491333, 0.0036201477, -0.08538818, -0.026382446, -0.10876465, -0.027557373, -0.03704834, 0.015823364, 0.03286743, 0.019348145, 0.0385437, -0.02368164, -0.022735596, 0.04876709, -0.01763916, 0.02053833, -0.016586304, 0.0059280396, 0.0079956055, 0.03829956, 0.0050086975, -0.028289795, 0.05569458, 0.00907135, -0.04611206, 0.0032787323, 0.005493164, 0.015945435, 0.036193848, -0.0057640076, -0.019546509, 0.015357971, 0.008514404, -0.023910522, 0.0027122498, 0

[1024, 1024, 1024]

In [13]:
get_embedding_dimensions(embed_v3_light, [string, string_2, string_3])

[[-0.1182251, -0.0098724365, 0.027862549, 0.032165527, -0.054260254, 0.076293945, 0.06317139, 0.018051147, 0.10235596, 0.009338379, 0.04019165, 0.018814087, 0.00048184395, -0.004627228, -0.03100586, -0.05404663, -0.023483276, 0.072143555, -0.09942627, -0.0011129379, -0.0030517578, -0.070251465, -0.02243042, 0.001490593, -0.08544922, 0.025558472, 0.017974854, 0.004398346, -0.06939697, -0.07513428, -0.055877686, 0.00041675568, 0.095825195, -0.0284729, 0.02218628, -0.01285553, -0.031341553, 0.026260376, 0.048736572, 0.040863037, -0.034423828, -0.11212158, 0.033569336, -0.01612854, -0.01361084, 0.030273438, 0.0027599335, -0.026916504, 0.12072754, 0.014076233, -0.027450562, -0.062805176, 0.009422302, 0.050567627, 0.03189087, 0.033569336, -0.04751587, 0.060943604, 0.027557373, 0.04510498, -0.06756592, 0.033203125, -0.091674805, 0.15307617, 0.17712402, 0.07897949, 0.02909851, -0.017242432, -0.017654419, 0.006198883, -0.024810791, 0.0029697418, -0.021072388, 0.007896423, 0.015342712, -0.020248

[384, 384, 384]

In [14]:
get_embedding_dimensions(embed_v2, [string, string_2, string_3])

[[1.7304688, 0.6640625, -0.22021484, -0.6015625, -2.2949219, -1.328125, 1.2392578, -3.2441406, 0.9824219, 1.8671875, 0.52197266, 0.32348633, -0.47680664, -1.96875, -3.2070312, 1.6113281, 1.5380859, -0.69140625, 0.8925781, -2.0214844, -0.96972656, 0.45239258, 0.6948242, 1.5166016, 0.22790527, 0.3828125, 2.4511719, 0.75878906, -0.5756836, 2.8535156, -1.3144531, 0.7709961, 1.5, 0.9824219, -1.0488281, -0.7089844, 0.94091797, -0.41870117, -1.4365234, 1.1289062, -1.9189453, 1.8759766, -0.033599854, -1.8115234, -0.08709717, -0.7636719, -2.5625, 1.2304688, -0.6118164, -1.1523438, -1.0361328, 0.5756836, 2.6640625, 0.22875977, 0.44921875, 1.1386719, -3.5195312, -3.0800781, 0.44067383, -0.21398926, 0.6645508, 3.0898438, -1.1503906, 1.0947266, 0.38134766, 0.64941406, -2.1933594, -3.1816406, -1.3222656, -0.3564453, 4.875, -0.43286133, 0.27612305, 0.67333984, 1.5976562, 0.32421875, -0.04714966, 0.83154297, 0.9121094, 0.10211182, -0.15686035, -1.7900391, -0.15014648, 0.76123047, -2.5839844, 0.3901367

[4096, 4096, 4096]

In [15]:
# Calculate similarity between two embeddings
# similarity() computes how similar two vectors are
# mode="cosine" uses cosine similarity (ranges from -1 to 1, where 1 = identical)
# Higher similarity scores indicate more semantically similar texts
# This is the core mechanism behind semantic search in RAG
embed_v3.similarity(
    embed_v3.get_text_embedding("""In embracing both the wisdom of the Sikh Gurus and the Stoic philosophers, 
                              we find a path to tranquility by accepting what is beyond our control and focusing 
                              our efforts on living virtuously and with purpose."""), 
    embed_v3.get_text_embedding(string_2),
    mode="cosine"  # Cosine similarity: measures angle between vectors (0-1 scale)
)

np.float64(0.18940321498701687)

# Create an Index

First, let's get some data

In [ ]:
import requests

def load_text_from_url(url: str) -> str:
    """
    Fetches and returns the text content from the specified URL.

    Parameters:
    - url: The URL of the text file to fetch.

    Returns:
    - The text content of the file if the request is successful; otherwise, an error message.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()  # This will raise an HTTPError if the response was an error
        return response.text
    except requests.RequestException as e:
        return f"Failed to load content from {url}. Error: {e}"

url = "https://www.gutenberg.org/files/10763/10763.txt"

text_content = load_text_from_url(url)

⏳ Generating embeddings can be time-consuming, especially with large volumes of text, due to numerous API calls required. 

Now, create an index by passing a **list of Documents**. To save time, and cost, we will only use 10,000 characters of the document

In [17]:
# Import Document and VectorStoreIndex from LlamaIndex core
# Document: Container for text content and metadata
# VectorStoreIndex: Index that stores documents as vector embeddings for semantic search
from llama_index.core import Document, VectorStoreIndex

# Create a Document with the full text content
# This contains the entire downloaded text
full_document = Document(text=text_content)

# Create a Document with a subset of the text (characters 50,000-60,000)
# Using a smaller subset saves time and API costs during indexing
# In production, you'd typically index the full document
partial_document = Document(text=text_content[50000:60000])

The `VectorStoreIndex` in LlamaIndex can be created in two ways: `from_documents` and `from_vector_store`.

- `from_documents`: when you have a set of documents that you want to index. This method takes these documents, computes their embeddings, and stores them in the vector store. 

- `from_vector_store`: when you already have computed embeddings that are stored in an external vector store (like Qdrant). This method connects to the external vector store and uses the pre-computed embeddings for the index. 



In [18]:
index = VectorStoreIndex.from_documents(
    # remember, you must pass a list of documents!
    [partial_document], 
    embed_model=embed_v3,
    show_progress=True)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Some nodes are missing content, skipping them...


Note, you can also build an index over a **list of `Node` objects**.


In [19]:
# Import SentenceSplitter for custom document chunking
# This gives you more control over how documents are split into nodes
from llama_index.core.node_parser import SentenceSplitter

# Instantiate a node parser with custom settings
# chunk_size: Maximum tokens per chunk (512 tokens = ~400 words)
# chunk_overlap: Overlapping tokens between chunks (maintains context)
# paragraph_separator: Prefer splitting at quadruple newlines (section breaks)
splitter = SentenceSplitter(
    chunk_size=512,  # Larger chunks than before (512 vs 128 tokens)
    chunk_overlap=16,  # Overlap to maintain context between chunks
    paragraph_separator="\n\n\n\n",  # Split at section breaks when possible
)

# Convert documents into nodes using the custom parser
# get_nodes_from_documents() splits the document(s) into smaller chunks
nodes = splitter.get_nodes_from_documents([partial_document])

# Create the index directly from pre-parsed nodes
# This approach gives you more control:
# 1. You've already split documents into nodes
# 2. Index will generate embeddings and store them
# Useful when you want custom chunking logic before indexing
index_from_nodes = VectorStoreIndex(
    nodes,  # Pass pre-parsed nodes instead of documents
    embed_model=embed_v3,  # Use the v3 embedding model
    show_progress=True  # Show progress bar during processing
)

Some nodes are missing content, skipping them...


Let's build on this pattern in the next lesson, where we'll store and persist our index for future use.